# 09 Hooks钩子与提交质量门


## 1. 对应课程与本篇范围

上一篇已经定义了测试专家 `tester` 和质量工程师 `quality-engineer`。本篇继续解决一个问题：如果主代理没有主动执行检查就准备提交代码，怎样在工具调用前增加一道检查。

对应网盘视频：

```text
20-Agent-自定义Hook钩子拦截工具调用.mp4
```

课程见 [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)，提取码 `1234`。主要来源是 [Part2 工程化控制和Agent开发](https://www.yuque.com/xxcls/vibecoding/ia0yu2m1v3sl537p) 的 4.3“Hooks 钩子系统”，以及网盘 `项目/黑马记账 APP/` 中的实际配置、提交代理和 Bash 检查脚本。

已读取讲义正文与原始 JSON 代码块、`.claude/settings.json`、`.claude/agents/gitcommit-agent.md`、`.claude/scripts/check-quality-gate.sh`。目录中还存在 `check-quality-gate.ps1`，但其正文未能预览，单文件下载提示失败；下文如提供 PowerShell 代码，会明确标为补充，不能当作课堂原文件。

视频未逐段观看或转录，项目快照也不代表视频每一步的版本。讲义中的事件名、环境变量和示例结构与当前官方接口有差异，下面先保留课程内容，再就地核对。当前行为依据 [Claude Code Hooks reference](https://code.claude.com/docs/en/hooks) 和 [Hooks guide](https://code.claude.com/docs/en/hooks-guide)，核对日期 `2026-10-10`。

本篇沿用 08 读取的 **Electron 项目快照**，不把它与 04 的 Tauri 示例混为同一份工程。下一节开始 RAG 项目，本篇不展开该项目。


## 2. Hook：在事件发生时执行检查

讲义定义：“Hooks 允许在特定事件前后自动执行脚本。”Hook 可以理解为挂在某个事件上的处理逻辑：事件发生时，Claude Code 查找匹配配置，运行处理器，再按该事件的规则处理结果。

把 07、08 的关系接起来看：

| 机制 | 解决的问题 | 本课程例子 |
|---|---|---|
| Skill | 一件事应按什么步骤做 | 创建、执行测试并生成报告 |
| Subagent | 由哪个专项角色完成任务 | 测试专家、质量工程师 |
| Hook | 在什么事件上运行检查或采取动作 | 提交命令执行前检查结果文件 |

角色提示词可以要求“提交前先测试”，Hook 则把检查挂到相应工具事件上。能否阻止动作，取决于事件时机、处理器返回格式和实际覆盖的调用路径。

本篇质量门的目标是：当 Claude Code 准备通过匹配的终端工具执行 `git commit` 时，先核对测试和质量检查的结果，缺少结果或结果不通过就拒绝这次工具调用。

**补充：**Hook 的触发机制可以固定，Hook 内部仍可能是模型判断。`type: command` 运行程序；`type: agent` 使用有工具访问能力的模型验证条件。自动触发不代表判断一定正确，也不代表任何位置执行的 Git 提交都会被覆盖。


## 3. 讲义的事件名称与当前对应关系

讲义列出以下 Hook 类型：

| 讲义名称 | 原文触发时机 | 当前接口中接近的组织方式 |
|---|---|---|
| `pre-command` | 执行任意命令之前 | `PreToolUse`，匹配相关终端工具 |
| `post-command` | 执行任意命令之后 | `PostToolUse`；失败调用另有 `PostToolUseFailure` |
| `on-start` | Claude Code 会话启动时 | `SessionStart` |
| `on-complete` | Claude Code 工作完成时 | 按目标选择 `Stop` 或其它事件 |
| `pre-edit` | 编辑文件之前 | `PreToolUse`，匹配 `Edit`、`Write` 等工具 |
| `post-edit` | 编辑文件之后 | `PostToolUse`，匹配相应文件工具 |

**当前写法差异：**右列是帮助理解的对应关系，左列不是当前官方事件名，不能只把左列名称复制进配置后假定生效。工具事件围绕 Claude Code 的工具调用，不是操作系统里所有命令或所有文件变化。

`Stop` 是主代理结束一轮响应时的事件，不等于应用退出，也不证明业务任务正确完成；会话结束对应 `SessionEnd`。子代理完成对应 `SubagentStop`。编辑也可能通过终端或其它工具完成，只匹配 `Edit|Write` 并不会覆盖所有写入方式。

本篇实际拦截点使用 **`PreToolUse`**。提交已经发生之后再检查，无法把这次提交变成“尚未执行”。


## 4. 讲义原始配置：先理解意图，再核对结构

讲义 4.3 给出的 JSON 原例：

```json
{
  "hooks": {
    "pre-command": {
      "command": "echo 'Claude 即将执行命令'",
      "description": "记录所有即将执行的命令"
    },
    "post-command": {
      "command": "./scripts/check-breaking-changes.sh",
      "description": "检查是否引入了破坏性变更"
    },
    "on-start": {
      "command": "git fetch --all",
      "description": "启动时拉取最新远程分支"
    }
  }
}
```

原例表达三种用途：执行前记录、执行后检查、会话启动时刷新远端引用。`git fetch --all` 更新远端引用，不会自动合并远端分支到当前工作区；`check-breaking-changes.sh` 只是原例中的脚本路径，其实现没有在这一段给出。

**当前写法差异：**官方接口使用实际事件名及数组结构，每个事件下是匹配组，每组里再放处理器数组。处理器需声明 `type`；不能把上面的事件对象和 `description` 原样当成当前有效配置。

本篇保留原例用于理解，不把它安装到当前电脑，也不执行其中命令。


## 5. 当前配置结构：事件、匹配组和处理器

下面是结构说明用的**补充示例**：在 Claude Code 使用 `Edit` 或 `Write` 成功操作文件后运行一个项目脚本。脚本路径是示意，必须先有相应文件才可使用。

```json
{
  "hooks": {
    "PostToolUse": [
      {
        "matcher": "Edit|Write",
        "hooks": [
          {
            "type": "command",
            "command": "node",
            "args": ["${CLAUDE_PROJECT_DIR}/.claude/scripts/check-style.cjs"],
            "timeout": 10
          }
        ]
      }
    ]
  }
}
```

| 层次或字段 | 含义 |
|---|---|
| 外层 `hooks` | 所有事件配置 |
| `PostToolUse` | 事件名，大小写按官方名称 |
| `matcher` | 此类工具事件匹配工具名，例中是 `Edit` 或 `Write` |
| 内层 `hooks` | 匹配后的处理器列表 |
| `type` | 处理器类型，本例是执行程序的 `command` |
| `command`、`args` | 执行程序及分开的参数 |
| `timeout` | Hook 超时，单位是秒 |

当前文档支持 `command`、`http`、`mcp_tool`、`prompt` 和 `agent` 等类型，各事件支持范围不同。本篇只展开原项目使用的 `agent` 和与原脚本对应的 `command`，不扩展其它集成。

常用位置：项目共享规则放 `.claude/settings.json`；本项目个人配置放 `.claude/settings.local.json`；跨项目个人规则放 `~/.claude/settings.json`。不同来源的 Hook 可以叠加，不能假定项目写了同一事件就替换掉个人配置的处理器。

修改时应合并到已有配置，并在 `/hooks` 中核对实际来源，避免覆盖项目其它设置。


## 6. Hook 怎样接收本次事件的数据

讲义列出的环境变量如下：

| 讲义变量 | 讲义含义 |
|---|---|
| `CLAUDE_EVENT_TYPE` | 事件类型 |
| `CLAUDE_COMMAND` | 将要执行或已执行的命令 |
| `CLAUDE_FILE_PATH` | 正在编辑的文件 |
| `CLAUDE_WORKING_DIR` | 当前工作目录 |

**当前接口差异：**官方协议主要把事件上下文作为 **JSON 输入**传给处理器。`command` Hook 从标准输入 stdin 读取，不能假定上面四个变量都会由当前 Claude Code 自动提供。

一个简化的 `PreToolUse` 输入示例：

```json
{
  "session_id": "example-session",
  "cwd": "/path/to/ledger-project",
  "hook_event_name": "PreToolUse",
  "tool_name": "Bash",
  "tool_input": {
    "command": "git commit -m \"修复分类筛选\""
  },
  "tool_use_id": "example-tool-call"
}
```

这里命令位于 `tool_input.command`；`Edit`、`Write` 等文件工具的目标路径通常位于 `tool_input.file_path`。字段取决于事件和工具，不应把所有输入都当成有 `command` 的终端调用。

`${CLAUDE_PROJECT_DIR}` 是当前文档支持的项目路径占位符，也会提供给处理器环境，用于定位脚本。但它指向会话启动项目，`cwd` 表示事件发生时的工作目录，两者可能不同，尤其是切换目录或进入 worktree 后。

在带 `args` 的 exec 形式中，路径作为一个独立参数传递，有空格也不需要额外添加引号字符；省略 `args` 的 shell 形式则必须按对应 Shell 正确引用路径。不要将事件中的命令字符串再次当作 Shell 代码执行来“解析”。


## 7. 返回结果：退出码与 JSON 各有适用位置

对本篇使用的**同步 `PreToolUse` command Hook**，最简单的方式是通过退出码表达检查结果：

| 处理结果 | 行为 |
|---|---|
| `exit 0`，不返回决策 JSON | Hook 检查成功，继续正常权限处理 |
| `exit 2` | 拒绝当前工具调用，stderr 提供拒绝原因 |
| 其它非零码，仅普通文本输出 | 一般作为非阻塞 Hook 错误，不等同于拒绝 |

`exit 0` 本身不等于已经执行了提交，也不自动授予提交权限。需要结构化控制时，command Hook 可输出类似下面的拒绝 JSON：

```json
{
  "hookSpecificOutput": {
    "hookEventName": "PreToolUse",
    "permissionDecision": "deny",
    "permissionDecisionReason": "缺少本次测试或质量检查结果"
  }
}
```

使用 JSON 时，stdout 应是符合该事件协议的有效 JSON，不要混入普通日志；本篇补充脚本选择 stderr + 退出码方式，避免混淆。

**事件边界：**`PostToolUse` 的工具已经执行，退出码 2 不能撤销它；`Stop` 的阻塞表示要求模型继续工作，不是拒绝某条 Git 命令。agent/prompt Hook 的模型响应又有自己的格式，不能直接套用 command Hook JSON。

**故障边界：**当前文档说明，`PreToolUse` 的 command Hook 超时通常不会阻塞原工具，输出被丢弃后继续正常权限流程。脚本找不到、解释器失败、超时等情况必须单独验证，不能把“配置了质量门”视为任何故障下都能拦截。


## 8. 原项目的质量门由哪些文件组成

本篇读取和目录核对的相关结构：

```text
黑马记账 APP/
└── .claude/
    ├── settings.json
    ├── agents/
    │   ├── gitcommit-agent.md
    │   ├── tester.md
    │   └── quality-engineer.md
    ├── scripts/
    │   ├── check-quality-gate.sh
    │   └── check-quality-gate.ps1
    └── results/
        ├── tester-result.txt       # 运行时由测试角色产生
        └── quality-result.txt      # 运行时由质量角色产生
```

结果文件是流程约定的运行产物，不代表本次已经生成。08 已读取两个检查角色：测试专家调用 `unit-test`，质量工程师组合 `security-audit`、`comments-check` 和额外质量规则。

本篇实际流程是：

```text
请求提交
  → gitcommit-agent 清理旧结果
  → tester 执行测试并写结果
  → quality-engineer 检查并写结果
  → 提交代理读取两份结果
  → 准备调用 git commit
  → PreToolUse Hook 再次核对结果
  → 满足条件后，提交命令才进入后续执行流程
```

提交代理负责组织工作，Hook 负责匹配调用前的再检查。Hook 本身只看结果时，并没有重新运行单元测试或安全审计。

**快照要区分：**目录中的 Bash 脚本说明自己供 Hook 调用，但最终 `settings.json` 实际配置的是 `type: agent`，没有引用这个 `.sh` 或 `.ps1`。因此“脚本存在”和“当前配置正在调用脚本”是两件事。


## 9. gitcommit-agent：组织两项检查

原 `.claude/agents/gitcommit-agent.md` 的头部：

```yaml
---
name: gitcommit-agent
description: Git 提交质量守门员——在提交代码前自动运行单元测试和代码质量检查，两者全部通过才允许提交并推送。当用户说"帮我提交代码"、"提交"、"commit"等时自动调用。
model: sonnet
tools: Agent, Read, Bash, Write, Glob
---
```

原工作流程首先清理旧通行证，再创建结果目录：

```bash
rm -f .claude/results/tester-result.txt .claude/results/quality-result.txt
mkdir -p .claude/results
```

这些是原应用项目根目录中的教学命令，本次未执行。原代理随后使用 `Agent` 工具按以下任务调用两个角色：

```text
subagent_type: "tester"
prompt: "质量门模式 — 请运行项目的所有单元测试（npx vitest run），并将结果写入 .claude/results/tester-result.txt。如果所有测试通过写 PASS，否则写 FAIL 并列出失败明细。"
```

```text
subagent_type: "quality-engineer"
prompt: "质量门模式 — 请对 src/ 目录进行全面的代码质量检查（安全审计 + 注释检查 + 代码质量审查），并将结果写入 .claude/results/quality-result.txt。三个维度全部通过写 PASS，否则写 FAIL 并列出问题。"
```

“质量门模式”关键词与 08 的两个角色定义对应，要求完成任务后写入结果，而不只是返回对话报告。原流程按第 2、3 步依次组织，没有要求两个角色必须并行。

**补充：**自定义代理正文、工具能力和当前版本都要支持这样的委派层次；主会话也可以按同样顺序组织。这里是在解释原文件，没有启动实际子代理。

原质量检查调用指定 `src/`，不能据此宣称已全面检查 `electron/` 或所有配置文件；要核对实际传递范围。注释技能的 30% 目标和质量角色 Gate Mode 的 20% 阈值差异沿用 08 的说明，正式使用时需要明确统一口径。


## 10. 提交代理怎样消费结果与处理成功、失败

原代理读取两份结果：

```bash
cat .claude/results/tester-result.txt
cat .claude/results/quality-result.txt
```

判定规则为：两个文件第一行都是 `PASS` 才通过；任何一个为 `FAIL` 或不存在都失败。其它非 PASS 内容也不应被当作通过。

原成功分支的操作依次是暂存、生成中文提交信息、提交、推送、销毁结果：

```bash
git add .
git log --oneline -5
git commit -m "提交信息

Co-Authored-By: Claude <noreply@anthropic.com>"
git push
rm -f .claude/results/tester-result.txt .claude/results/quality-result.txt
```

原失败分支要求向用户展示详情、销毁通行证，再告知修复后重新运行。重要规则是两项检查都执行、开始先清理旧结果、结束后无论 PASS/FAIL 都清理，并明确报告每一步状态。

**补充：**这是原角色定义的完整意图，用户只要求学习笔记，不构成本次运行 `git add`、提交或推送的指令。实际使用代理时仍要遵守用户本次授权范围；要求提交本地改动不自动意味着要求推送远端。

`git add .` 会暂存未被忽略的结果文件和其它改动，可能把 `.claude/results/` 的通行证也加入提交。该运行目录应按项目约定忽略，并检查暂存 diff；本次没有读取到 `.gitignore` 正文，不能声称原项目已正确排除这些文件。

推送失败时应分别报告“本地提交成功”和“远端同步失败”，不能合成整体成功。结果清理也应覆盖异常退出路径，不能只考虑脚本按预期走完的情况。


## 11. 原 Bash 检查脚本：两份第一行都必须是 PASS

原文件是 `.claude/scripts/check-quality-gate.sh`。下面保留核心判断及命令，省略重复的装饰性 `echo`；这是原脚本摘录，不是当前接口修正版。

```bash
RESULTS_DIR=".claude/results"
TESTER_FILE="$RESULTS_DIR/tester-result.txt"
QUALITY_FILE="$RESULTS_DIR/quality-result.txt"

if [ ! -f "$TESTER_FILE" ] || [ ! -f "$QUALITY_FILE" ]; then
  echo "  原因：没有找到单元测试或代码质量的通行证。"
  exit 2
fi

TESTER_RESULT=$(head -1 "$TESTER_FILE")
QUALITY_RESULT=$(head -1 "$QUALITY_FILE")

if [ "$TESTER_RESULT" = "PASS" ] && [ "$QUALITY_RESULT" = "PASS" ]; then
  echo "  单元测试 : PASS"
  echo "  代码质量 : PASS"
  exit 0
else
  if [ "$TESTER_RESULT" != "PASS" ]; then
    echo "  [单元测试] FAIL"
    cat "$TESTER_FILE"
    echo ""
  fi
  if [ "$QUALITY_RESULT" != "PASS" ]; then
    echo "  [代码质量] FAIL"
    cat "$QUALITY_FILE"
    echo ""
  fi
  echo "  请修复上述问题后重新运行质量检查。"
  exit 2
fi
```

脚本没有执行 Vitest，只读取两份文件；`head -1` 取第一行，因此 FAIL 后的多行详情不会改变第一行判定。只有两个精确的 `PASS` 字符串同时满足时才返回 0。

对照当前协议，有三个需要注意的地方：

1. 原失败提示通过 `echo` 写到 stdout，`PreToolUse` 退出码 2 的拒绝说明应通过 stderr 提供，否则用户和模型可能收不到预期的详细原因。
2. `.claude/results` 是相对路径，工作目录改变后可能检查错位置。
3. Windows CRLF 第一行可能让 Bash 读到 `PASS\r`，与 `PASS` 不相等；空白、编码和读取失败也需要明确处理。

这些是根据已读取代码能判断的边界，本次未运行脚本验证。


## 12. 原 settings.json：最终快照使用 agent Hook

下面只摘录原 `.claude/settings.json` 的 `hooks` 部分；其他内容主要是课堂机器的权限和目录记录，不作为通用权限模板复制。

```json
{
  "hooks": {
    "PreToolUse": [
      {
        "matcher": "Bash",
        "hooks": [
          {
            "type": "agent",
            "prompt": "You are a git commit quality gate. Use Read tool to check two files: .claude/results/tester-result.txt and .claude/results/quality-result.txt. If BOTH files exist AND BOTH first lines are exactly 'PASS': return {\"hookSpecificOutput\": {\"permissionDecision\": \"allow\"}, \"systemMessage\": \"Quality gate PASS - commit allowed\"}. Otherwise: return {\"hookSpecificOutput\": {\"permissionDecision\": \"deny\"}, \"systemMessage\": \"Quality gate BLOCKED - run gitcommit-agent first to generate pass markers at .claude/results/\"}. You MUST actually read the files with Read tool before deciding.",
            "if": "Bash(*git commit*)",
            "timeout": 30
          }
        ]
      }
    ]
  }
}
```

逐层读这段配置：

| 配置 | 作用 |
|---|---|
| `PreToolUse` | 工具实际执行前检查 |
| `matcher: Bash` | 只进入 Bash 工具调用这一类 |
| `if: Bash(*git commit*)` | 当前文档支持的权限规则形式过滤条件，进一步选择相关命令 |
| `type: agent` | 启动验证用代理，让它实际读文件 |
| `prompt` | 要求两个文件存在且第一行都为 PASS |
| `timeout: 30` | 本例设置的 Hook 超时秒数 |

**当前接口差异：**agent Hook 的模型响应应为 `{"ok": true}`，或 `{"ok": false, "reason": "..."}`。原 prompt 要求返回 `hookSpecificOutput.permissionDecision`，混用了 command Hook 的事件输出格式，而且没有其事件名字段；不能把这段原配置直接当作已验证有效的当前实现。

当前文档还将 agent Hook 标为实验能力。它与 `.claude/agents/gitcommit-agent.md` 不是同一个角色：提交代理组织测试和提交，Hook 验证代理在事件上检查结果。


## 13. 补充：沿用 agent Hook 时怎样修正响应约定

如果继续使用原项目的 agent 路线，至少要把 prompt 的返回约定改成该类型的协议，并明确结果属于哪个项目。下面是与原配置对应的补充示例，不是已安装配置：

```json
{
  "hooks": {
    "PreToolUse": [
      {
        "matcher": "Bash",
        "hooks": [
          {
            "type": "agent",
            "if": "Bash(*git commit*)",
            "prompt": "检查此次提交所在应用项目的 .claude/results/tester-result.txt 和 .claude/results/quality-result.txt。事件输入：$ARGUMENTS。必须实际读取两份文件；只有两个文件都可读且第一行都恰好为 PASS 时返回 {\"ok\": true}。缺失、读取失败或任一第一行不是 PASS 时返回 {\"ok\": false, \"reason\": \"说明具体失败项，并要求重新完成质量检查\"}。只验证，不改写结果文件，不提交代码。",
            "timeout": 30
          }
        ]
      }
    ]
  }
}
```

`$ARGUMENTS` 在 prompt/agent Hook 中表示本次事件输入 JSON，与 07 技能中同名参数变量的使用场景不同。验证代理可以据此了解工作目录和待执行工具，但不要让它修改 PASS 文件来满足自己的条件。

这种修正解决了返回格式，不会自动解决过期结果、项目定位或提交命令识别的全部问题。只读取固定文件的判断很简单，也可采用下一节的 command 路线，直接用程序表达条件。

两种路线作为备选实现理解即可；不要把旧 agent Hook 与新 command Hook 同时叠加后，再误以为只会运行一个检查。


## 14. 补充：用同步 command Hook 对接 Bash 检查

下面把原脚本的判断整理成与当前协议配合的短版本。保存目标仍是**应用项目中的** `.claude/scripts/check-quality-gate.sh`；本次只是 Notebook 中的示例，没有改写原应用脚本。

```bash
#!/usr/bin/env bash

# 按脚本自身位置定位结果，避免依赖调用时的工作目录。
SCRIPT_DIR="$(cd -- "$(dirname -- "${BASH_SOURCE[0]}")" && pwd)" || exit 2
RESULTS_DIR="$SCRIPT_DIR/../results"

for result_file in "$RESULTS_DIR/tester-result.txt" "$RESULTS_DIR/quality-result.txt"; do
  if [ ! -f "$result_file" ] || [ ! -r "$result_file" ]; then
    printf '质量门拒绝：结果文件不存在或不可读：%s\n' "$result_file" >&2
    exit 2
  fi

  first_line=$(head -n 1 "$result_file") || {
    printf '质量门拒绝：读取结果失败：%s\n' "$result_file" >&2
    exit 2
  }
  # 允许 Windows CRLF 行结束符；不删除 PASS 前后的普通空格。
  first_line=${first_line%$'\r'}

  if [ "$first_line" != "PASS" ]; then
    printf '质量门拒绝：第一行不是 PASS：%s\n' "$result_file" >&2
    cat "$result_file" >&2
    exit 2
  fi
done

# 不输出 allow，保留正常权限处理。
exit 0
```

对应的当前配置示例：

```json
{
  "hooks": {
    "PreToolUse": [
      {
        "matcher": "Bash",
        "hooks": [
          {
            "type": "command",
            "if": "Bash(*git commit*)",
            "command": "bash",
            "args": ["${CLAUDE_PROJECT_DIR}/.claude/scripts/check-quality-gate.sh"],
            "timeout": 10
          }
        ]
      }
    ]
  }
}
```

显式调用 `bash`，脚本路径作为单独参数，支持应用目录里的空格。环境须能找到 Bash 及 `dirname`、`head` 等命令；Windows 上通常对应已有 Git Bash 环境。

本例按脚本所在检出目录定位结果。若会话在另一 worktree 中提交，应让脚本与结果都对应那份检出目录，不能继续拿主检出的 PASS 使用。原课并未实现 worktree 结果绑定，本篇也不把它说成已经具备。

这里采用同步检查，没有设置 `async: true`。异步 Hook 的原工具可能已继续执行，不能作为这次提交前的阻塞检查。


## 15. 补充：原生 PowerShell 工具需要单独覆盖

原快照只有 `matcher: Bash`。当前 Claude Code 在部分 Windows 环境中使用 `PowerShell` 工具，只匹配 Bash 就不会触发。终端工具的名称与 Hook 使用什么解释器也是两回事：匹配 PowerShell 调用的 Hook，仍可启动任意已配置的检查程序。

下面是按同一判断逻辑整理的 **PowerShell 补充版本**，不是本次未读到的原 `.ps1` 正文。目标文件为应用项目 `.claude/scripts/check-quality-gate.ps1`：

```powershell
$resultsDir = Join-Path $PSScriptRoot '../results'

foreach ($resultName in @('tester-result.txt', 'quality-result.txt')) {
    $resultPath = Join-Path $resultsDir $resultName
    try {
        $firstLine = Get-Content -LiteralPath $resultPath -TotalCount 1 -ErrorAction Stop
    }
    catch {
        [Console]::Error.WriteLine("质量门拒绝：结果文件缺失或不可读：$resultPath")
        exit 2
    }

    # -cne 区分大小写，空文件或其它文本同样拒绝。
    if ($firstLine -cne 'PASS') {
        [Console]::Error.WriteLine("质量门拒绝：第一行不是 PASS：$resultPath")
        exit 2
    }
}

exit 0
```

在同一 `PreToolUse` 配置中，针对实际存在的 PowerShell 工具增加下面的匹配组，与 Bash 组并列：

```json
{
  "matcher": "PowerShell",
  "hooks": [
    {
      "type": "command",
      "if": "PowerShell(*git commit*)",
      "command": "powershell.exe",
      "args": [
        "-NoProfile",
        "-File",
        "${CLAUDE_PROJECT_DIR}/.claude/scripts/check-quality-gate.ps1"
      ],
      "timeout": 10
    }
  ]
}
```

`powershell.exe` 需要在当前环境可用，脚本也要符合机器已有的执行策略。本篇没有为了运行示例修改执行策略或其它权限。解释器未启动和脚本明确返回 2 是不同状态，必须通过实际 Hook 日志和阻塞行为区分。

PowerShell 默认字符串比较通常不区分大小写，因此这里明确使用 `-cne`。两种补充实现都不应通过自动改写结果文件来“修复”缺少 PASS 的问题。


## 16. 结果有效期：清理旧文件还不够

原提交代理用“开始先清理、结束后销毁”避免重复使用通行证。这能减少旧结果残留，但固定两个文本文件仍存在有效期问题。

例如：测试和审查都通过并写入 PASS，随后业务文件被修改，再执行提交。Hook 只看文件第一行时，仍会看到两个 PASS，但它们对应的是修改前的代码。

**补充：**有效结果至少要对应这一次目标项目、检查范围和将要提交的内容。正式完善时可记录检查对象的标识，并在提交前核对；这里只指出原实现没有完成的绑定，不额外扩展一套新的质量平台。

还要区分工作区、暂存区和提交内容：测试通常读取工作区，`git commit` 通常提交暂存区。如果暂存的是旧版本、工作区包含未暂存修复，工作区测试通过也不能直接证明暂存版本通过。

因此消费结果前，应保证检查后相关内容没有变化，并核对将要提交的 diff。若变化了，就重新检查，不手动写 `PASS` 跳过流程。结果文件本身也只是可修改文本，不是不可伪造的证明。


## 17. 补充：验证拦截是否真正生效

先分别验证脚本、Hook 加载和事件匹配，再验证与提交代理配合。以下是学习检查项，本次没有在应用项目执行。

| 输入或场景 | 应观察的结果 |
|---|---|
| 两份结果文件都缺失 | 脚本返回 2；匹配的工具调用被拒绝 |
| 只存在一份 PASS | 拒绝，并指出另一份缺失 |
| 一份 PASS、一份 FAIL | 拒绝，不执行提交命令 |
| 空文件、`pass` 或带普通空格的第一行 | 补充脚本拒绝，不将相近文本当作 PASS |
| 两份第一行都为 PASS | 脚本返回 0，继续正常权限流程 |
| 文件使用 CRLF | Bash 补充版本去除行尾 CR 后按同一规则判断 |
| 非提交命令 | 不应被误当成需要两份 PASS 的提交 |
| 从 Bash 与 PowerShell 各发起匹配调用 | 只在对应匹配组正确加载时触发 |
| 检查脚本缺失或解释器不可用 | 出现 Hook 错误；不能据此宣布质量门正常 |
| 修改代码后保留旧 PASS | 原简单实现无法识别过期，应重新检查并说明限制 |

检查脚本本身可以用命名的临时测试目录和固定样例验证，不必通过真实提交制造测试。查看返回码时要立即读取，避免被后续命令覆盖：

```bash
bash .claude/scripts/check-quality-gate.sh
gate_status=$?
printf '质量门退出码：%s\n' "$gate_status"
```

```powershell
powershell.exe -NoProfile -File '.claude/scripts/check-quality-gate.ps1'
$gateStatus = $LASTEXITCODE
Write-Output "质量门退出码：$gateStatus"
```

上述命令应在相应应用根目录使用。脚本的 0/2 返回正确，只证明脚本判定路径；还要在 Claude Code 中观察 `PreToolUse` 是否真正触发，拒绝时目标工具是否确实未执行。

手动终端调用脚本不会经过 Claude Code 的工具事件。学习验证时也不要为了制造“通过”把样例 PASS 写进真实项目的结果目录。


## 18. 补充：加载与排查，先找具体失败层

当前文档中的 `/hooks` 是查看已配置 Hook 的只读入口，可以看事件、处理内容和来源。编辑配置后，当前版本通常会监测文件变化；必要时重启，再核对实际加载内容，不能只看文件“已经保存”。

| 现象 | 优先核对 |
|---|---|
| 没触发 | 实际工具名、事件大小写、matcher、`if` 条件和来源 |
| 修改后仍像旧逻辑 | 是否有个人或其它来源的同类 Hook 叠加，当前会话是否已加载 |
| 脚本找不到 | 项目占位符、脚本文件名、参数路径和解释器 |
| 找不到结果文件 | 实际项目、工作目录、脚本定位方式和检查角色是否完成写入 |
| 有输出却没拒绝 | 是否在 `PreToolUse`，是否返回 2 或正确 JSON，是否用了异步 |
| 拒绝但没有原因 | 失败信息是否写到 stderr，返回协议是否符合处理器类型 |
| agent Hook 格式报错 | 是否仍使用原 prompt 中的 `permissionDecision` 而非 `ok` |
| 一直等待或超时 | 检查是否包含耗时任务；文件核对不应顺便运行全套测试 |

当前文档建议通过 `claude --debug` 查看 `~/.claude/debug/<session-id>.txt`，或用 `--debug-file` 指定日志文件。例如以下是终端启动命令，路径为学习示例：

```bash
claude --debug-file ./claude-hook-debug.log
```

日志用于确认匹配、启动、退出状态和解析错误，不代表测试结果本身。当前 `--debug` 写日志而不是直接把所有日志打印到终端，遇到版本差异时以实际帮助信息为准。

如果改用 `Stop` 检查完成条件，需留意 `stop_hook_active`，避免同一个永远无法满足的条件让模型反复继续。本课程提交门使用 `PreToolUse`，无需把所有检查搬到 Stop。


## 19. 提交质量门的覆盖边界

原项目把 Hook 挂在 Claude Code 的工具调用上，因此它与 Git 自身的 `pre-commit` Hook 不是同一机制。

| 路径或情况 | 本篇实现能说明什么 |
|---|---|
| Claude Code 经已匹配工具执行提交命令 | 可以在调用前检查两份结果 |
| 用户在外部终端执行 `git commit` | 不会因此触发 Claude Code 的 `PreToolUse` |
| 其它 Git GUI 或外部程序提交 | 同样不能假定被覆盖 |
| 通过包装脚本、别名或复杂命令执行提交 | 简单 `if` 条件不等于完整 Shell 语义识别 |
| Hook 程序异常、超时或加载失败 | 需要单独检查故障行为，不能当作检查通过 |
| 结果是 PASS，但之后代码又变了 | 两个固定标记无法单独证明结果仍然有效 |

当前文档明确说明 `if` 命令过滤是尽力匹配，不能把一条字符串模式当成全面的命令权限边界。生产流程如果需要统一约束全部提交或合入，还需要对应入口上的机制；本篇不额外实现这些系统。

原质量门的教学价值，是把“角色组织检查 → 写结构化结果 → 工具前消费结果”接起来。使用时应报告这一次真实检查的范围、结果和实际拦截路径，不把两份 PASS 等同于整个项目已经没有问题。


## 20. 来源与本次整理状态

课程资料：

- [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)：核对第 20 节准确标题及后续 RAG 章节边界。
- [Part2 工程化控制和Agent开发](https://www.yuque.com/xxcls/vibecoding/ia0yu2m1v3sl537p)：读取 4.3 事件、环境变量表和原始 JSON 代码块。
- 网盘 `项目/黑马记账 APP/.claude/settings.json`：读取全文，正文只保留与本篇相关的 Hook 原配置。
- 同项目 `.claude/agents/gitcommit-agent.md`：读取源码模式全文，整理清理、委派、判定、提交、推送和结果销毁的原流程。
- 同项目 `.claude/scripts/check-quality-gate.sh`：读取全文，保留核心逻辑并说明输出和路径边界。
- 同目录 `.ps1` 仅核对存在，未取得正文；PowerShell 示例与 Bash 修正版均明确属于补充。

接口核对：

- [Hooks reference](https://code.claude.com/docs/en/hooks)：当前事件、配置、stdin JSON、退出码与输出协议、agent Hook 格式、路径、工具匹配及调试。
- [Automate actions with hooks](https://code.claude.com/docs/en/hooks-guide)：自动化用途与基本配置方式。核对日期 `2026-10-10`。

本次只新增 09 Notebook。没有把 Hook、脚本或代理安装到当前电脑，没有执行课程中的测试、清理结果文件、暂存、提交或推送，也没有修改前 8 篇笔记。文件中的配置、返回码和预期场景是学习内容，不是本次实际执行结果。
